# Synthetic Rank-One Test

In [109]:
import numpy as np
from ssl_shareability_metric.ssl_encoder_shareability import ssl_shareability
from ssl_shareability_metric.whiten_and_center import whiten_and_center
import torch
from torch.utils.data import TensorDataset, DataLoader
from ssl_shareability_metric.shared_encoder import SharedEncoder
from ssl_shareability_metric.seperate_encoder import SeperateEncoder
import copy

torch.manual_seed(42)

## Create rank 1 future representations for synthetic low mid and high sharability

In [110]:
rng = np.random.default_rng(42)

current = rng.normal(size=(5000, 13))

u_low = np.zeros(13)
v_low = np.zeros(13)

u_low[0] = 1.0
v_low[1] = 1.0

M_low = 0.9 * np.outer(u_low, v_low)

future_low_raw = current @ M_low.T

u_mid = np.zeros(13)
v_mid = np.zeros(13)

u_mid[0] = 1.0
v_mid[0] = 0.5
v_mid[1] = np.sqrt(0.75)

M_mid = 0.9 * np.outer(u_mid, v_mid)

future_mid_raw = current @ M_mid.T

u_high = np.zeros(13)
v_high = np.zeros (13)

u_high[0] = 1
v_high[0] = 1

M_high = 0.9 * np.outer(u_high, v_high)

future_high_raw = current @ M_high.T

noise_cov = np.eye(13) - M_low @ M_low.T

noise_eigenvalues, noise_eigenvectors = np.linalg.eigh(noise_cov)

noise_eigenvalues_sqrt = np.sqrt(noise_eigenvalues)

noise_sqrt = noise_eigenvectors @ np.diag(noise_eigenvalues_sqrt) @ noise_eigenvectors.T

raw_noise = rng.normal(size=current.shape)

noise = raw_noise @ noise_sqrt

future_low = future_low_raw + noise
future_mid = future_mid_raw + noise
future_high = future_high_raw + noise

print(f"current: {current.shape}")
print(f"future low: {future_low.shape}")
print(f"future mid: {future_mid.shape}")
print(f"future high: {future_high.shape}")

current: (5000, 13)
future low: (5000, 13)
future mid: (5000, 13)
future high: (5000, 13)


## Create splits

In [111]:
train_split_idx = int(current.shape[0] * 0.70)
val_split_idx = train_split_idx + int(current.shape[0] * 0.10)

train_current = current[:train_split_idx]
val_current = current[train_split_idx:val_split_idx]
test_current = current[val_split_idx:]

train_future_low = future_low[:train_split_idx]
val_future_low = future_low[train_split_idx:val_split_idx]
test_future_low = future_low[val_split_idx:]

train_future_mid = future_mid[:train_split_idx]
val_future_mid = future_mid[train_split_idx:val_split_idx]
test_future_mid = future_mid[val_split_idx:]

train_future_high = future_high[:train_split_idx]
val_future_high = future_high[train_split_idx:val_split_idx]
test_future_high = future_high[val_split_idx:]

print(f"train current: {train_current.shape}")
print(f"val current: {val_current.shape}")
print(f"test current: {test_current.shape}")

print(f"train future low: {train_future_low.shape}")
print(f"val future low: {val_future_low.shape}")
print(f"test future low: {test_future_low.shape}")

print(f"train future mid: {train_future_mid.shape}")
print(f"val future mid: {val_future_mid.shape}")
print(f"test future mid: {test_future_mid.shape}")

print(f"train future high: {train_future_high.shape}")
print(f"val future high: {val_future_high.shape}")
print(f"test future high: {test_future_high.shape}")

train current: (3500, 13)
val current: (500, 13)
test current: (1000, 13)
train future low: (3500, 13)
val future low: (500, 13)
test future low: (1000, 13)
train future mid: (3500, 13)
val future mid: (500, 13)
test future mid: (1000, 13)
train future high: (3500, 13)
val future high: (500, 13)
test future high: (1000, 13)


## Shareability metric

In [112]:
low_shareability, _, _ = ssl_shareability(train_current, train_future_low)
mid_shareability, _, _ = ssl_shareability(train_current, train_future_mid)
high_shareabilitty, _, _ = ssl_shareability(train_current, train_future_high)

print(f"low shareability: {low_shareability}")
print(f"mid shareability: {mid_shareability}")
print(f"high shareability: {high_shareabilitty}")

low shareability: 0.5237084247340593
mid shareability: 0.7611133262980686
high shareability: 0.9992489480932957


## Whiten and center

In [113]:
train_current_whitend_centered = whiten_and_center(train_current)
val_current_whitend_centered = whiten_and_center(val_current)
test_current_whitend_centered = whiten_and_center(test_current)

train_future_low_whitend_centered = whiten_and_center(train_future_low)
val_future_low_whitend_centered = whiten_and_center(val_future_low)
test_future_low_whitend_centered = whiten_and_center(test_future_low)

train_future_mid_whitend_centered = whiten_and_center(train_future_mid)
val_future_mid_whitend_centered = whiten_and_center(val_future_mid)
test_future_mid_whitend_centered = whiten_and_center(test_future_mid)

train_future_high_whitend_centered = whiten_and_center(train_future_high)
val_future_high_whitend_centered = whiten_and_center(val_future_high)
test_future_high_whitend_centered = whiten_and_center(test_future_high)

## Convert to tensros

In [114]:
train_current_tensor = torch.tensor(train_current_whitend_centered, dtype=torch.float32)
val_current_tensor = torch.tensor(val_current_whitend_centered, dtype=torch.float32)
test_current_tensor = torch.tensor(test_current_whitend_centered, dtype=torch.float32)

train_future_low_tensor = torch.tensor(train_future_low_whitend_centered, dtype=torch.float32)
val_future_low_tensor = torch.tensor(val_future_low_whitend_centered, dtype=torch.float32)
test_future_low_tensor = torch.tensor(test_future_low_whitend_centered, dtype=torch.float32)

train_future_mid_tensor = torch.tensor(train_future_mid_whitend_centered, dtype=torch.float32)
val_future_mid_tensor = torch.tensor(val_future_mid_whitend_centered, dtype=torch.float32)
test_future_mid_tensor = torch.tensor(test_future_mid_whitend_centered, dtype=torch.float32)

train_future_high_tensor = torch.tensor(train_future_high_whitend_centered, dtype=torch.float32)
val_high_tensor = torch.tensor(val_future_high_whitend_centered, dtype=torch.float32)
test_future_high_tensor = torch.tensor(test_future_high_whitend_centered, dtype=torch.float32)

train_low = TensorDataset(train_current_tensor, train_future_low_tensor)
val_low = TensorDataset(val_current_tensor, val_future_low_tensor)

train_mid = TensorDataset(train_current_tensor, train_future_mid_tensor)
val_mid = TensorDataset(val_current_tensor, val_future_mid_tensor)

train_high = TensorDataset(train_current_tensor, train_future_high_tensor)
val_high = TensorDataset(val_current_tensor, val_high_tensor)

train_low_dateloader = DataLoader(train_low, batch_size=64, shuffle=False)
val_low_dateloader = DataLoader(val_low, batch_size=64, shuffle=False)
train_mid_dateloader = DataLoader(train_mid, batch_size=64, shuffle=False)
val_mid_dateloader = DataLoader(val_mid, batch_size=64, shuffle=False)
train_high_dateloader = DataLoader(train_high, batch_size=64, shuffle=False)
val_high_dateloader = DataLoader(val_high, batch_size=64, shuffle=False)

## Init encoders and optimizers

In [115]:
low_shared_encoder = SharedEncoder(vector_size=13)
low_shared_optimizer = torch.optim.Adam(low_shared_encoder.parameters(), lr=1e-2)

low_seperate_encoder = SeperateEncoder(vector_size=13)
low_seperate_optimizer = torch.optim.Adam(low_seperate_encoder.parameters(), lr=1e-2)

## Low Rank 1 Case

### Shared

In [116]:
epochs = 100
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    low_shared_encoder.train()
    for x, y in train_low_dateloader:
        Z_x = low_shared_encoder(x)
        Z_y = low_shared_encoder(y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        low_shared_optimizer.zero_grad()
        loss.backward()
        low_shared_optimizer.step()
        with torch.no_grad():
            weights = low_shared_encoder.shared.weight
            weights.div_(weights.norm(p=2))
             
    val_loss = 0
    low_shared_encoder.eval()
    for x, y in val_low_dateloader:
        with torch.no_grad():
            Z_x = low_shared_encoder(x)
            Z_y = low_shared_encoder(y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_low_dateloader)} val: {val_loss / len(val_low_dateloader)}")   
            
    avg_val_loss = val_loss / len(val_low_dateloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        best_state = copy.deepcopy(low_shared_encoder.state_dict())

        
if best_state is not None:
    low_shared_encoder.load_state_dict(best_state)

epoch: 1 train: -0.19563175683671777 val: -0.22974276449531317
epoch: 2 train: -0.35386956388300117 val: -0.34910580702126026
epoch: 3 train: -0.42222147394310344 val: -0.39285712502896786
epoch: 4 train: -0.4421066858551719 val: -0.4098925553262234
epoch: 5 train: -0.44879990165883843 val: -0.4178099911659956
epoch: 6 train: -0.4514214819127863 val: -0.4219835065305233
epoch: 7 train: -0.45254294086586344 val: -0.42435091733932495
epoch: 8 train: -0.45302993899041955 val: -0.42574296332895756
epoch: 9 train: -0.4532210810617967 val: -0.42656815238296986
epoch: 10 train: -0.45326536406170237 val: -0.4270489439368248
epoch: 11 train: -0.4532343691045588 val: -0.4273147415369749
epoch: 12 train: -0.4531650811433792 val: -0.42744479700922966
epoch: 13 train: -0.45307753546671437 val: -0.42748891562223434
epoch: 14 train: -0.45298287976871837 val: -0.4274796489626169
epoch: 15 train: -0.4528875093568455 val: -0.42743771336972713
epoch: 16 train: -0.4527950584888458 val: -0.4273770209401846

### Seperate

In [117]:
epochs = 100
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    low_seperate_encoder.train()
    for x, y in train_low_dateloader:
        Z_x, Z_y = low_seperate_encoder(x, y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        low_seperate_optimizer.zero_grad()
        loss.backward()
        low_seperate_optimizer.step()
        with torch.no_grad():
            current_weights = low_seperate_encoder.current.weight
            future_weights = low_seperate_encoder.future.weight
            
            current_weights.div_(current_weights.norm(p=2)) 
            future_weights.div_(future_weights.norm(p=2))
             
    val_loss = 0
    low_seperate_encoder.eval()
    for x, y in val_low_dateloader:
        with torch.no_grad():
            Z_x, Z_y = low_seperate_encoder(x, y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_low_dateloader)} val: {val_loss / len(val_low_dateloader)}")   
            
    avg_val_loss = val_loss / len(val_low_dateloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        best_state = copy.deepcopy(low_seperate_encoder.state_dict())

        
if best_state is not None:
    low_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.10911167982796377 val: -0.13483379082754254
epoch: 2 train: -0.2220790112729777 val: -0.34839778766036034
epoch: 3 train: -0.556288665803996 val: -0.6900747716426849
epoch: 4 train: -0.7618335019458424 val: -0.7999728322029114
epoch: 5 train: -0.8200958143581044 val: -0.8285083100199699
epoch: 6 train: -0.8340059789744291 val: -0.8338066413998604
epoch: 7 train: -0.8347064245830883 val: -0.8316299393773079
epoch: 8 train: -0.831057835708965 val: -0.826856330037117
epoch: 9 train: -0.8259590062228116 val: -0.821217305958271
epoch: 10 train: -0.8204741055315191 val: -0.8153972513973713
epoch: 11 train: -0.815027327971025 val: -0.8096892274916172
epoch: 12 train: -0.8097974939779802 val: -0.804227989166975
epoch: 13 train: -0.8048622445626692 val: -0.7990772984921932
epoch: 14 train: -0.8002542354843833 val: -0.7942673973739147
epoch: 15 train: -0.7959845553744923 val: -0.7898095324635506
epoch: 16 train: -0.7920529040423306 val: -0.7857045829296112
epoch: 17 train: -0.

### Holdout Set

In [118]:
with torch.no_grad():
    Z_x = low_shared_encoder(test_current_tensor)
    Z_y = low_shared_encoder(test_future_low_tensor)
    low_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {low_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = low_seperate_encoder(test_current_tensor, test_future_low_tensor)
    low_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {low_sep_loss}")

shared loss: -0.44774556159973145
seperate loss: -0.8191031813621521


### Analysis

In [119]:
low_test_result = low_shared_loss / low_sep_loss
print(f"Low test result: {low_test_result}, Low shareability result: {low_shareability}")

Low test result: 0.546629011631012, Low shareability result: 0.5237084247340593


## Mid Rank One Case

In [120]:
mid_shared_encoder = SharedEncoder(vector_size=13)
mid_shared_optimizer = torch.optim.Adam(mid_shared_encoder.parameters(), lr=1e-2)

mid_seperate_encoder = SeperateEncoder(vector_size=13)
mid_seperate_optimizer = torch.optim.Adam(mid_seperate_encoder.parameters(), lr=1e-2)

In [121]:
epochs = 100
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    mid_shared_encoder.train()
    for x, y in train_mid_dateloader:
        Z_x = mid_shared_encoder(x)
        Z_y = mid_shared_encoder(y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        mid_shared_optimizer.zero_grad()
        loss.backward()
        mid_shared_optimizer.step()
        with torch.no_grad():
            weights = mid_shared_encoder.shared.weight
            weights.div_(weights.norm(p=2))
             
    val_loss = 0
    mid_shared_encoder.eval()
    for x, y in val_mid_dateloader:
        with torch.no_grad():
            Z_x = mid_shared_encoder(x)
            Z_y = mid_shared_encoder(y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_mid_dateloader)} val: {val_loss / len(val_mid_dateloader)}")   
            
    avg_val_loss = val_loss / len(val_mid_dateloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        best_state = copy.deepcopy(mid_shared_encoder.state_dict())
    
if best_state is not None:
    mid_shared_encoder.load_state_dict(best_state)
        

epoch: 1 train: -0.10385622445663267 val: -0.10598326753824949
epoch: 2 train: -0.11810956968163902 val: -0.0832790806889534
epoch: 3 train: -0.13664803513410417 val: -0.07155202003195882
epoch: 4 train: -0.1702027937058698 val: -0.1468043434433639
epoch: 5 train: -0.3866475915366953 val: -0.5449986793100834
epoch: 6 train: -0.6167614627968181 val: -0.623688917607069
epoch: 7 train: -0.6433428569273515 val: -0.6286133788526058
epoch: 8 train: -0.6421411736444993 val: -0.6267258673906326
epoch: 9 train: -0.6390021600506522 val: -0.6244835183024406
epoch: 10 train: -0.6362955602732572 val: -0.6225918121635914
epoch: 11 train: -0.6341344193978743 val: -0.6210616119205952
epoch: 12 train: -0.6324035687880083 val: -0.6198203153908253
epoch: 13 train: -0.6309947783296759 val: -0.6187995970249176
epoch: 14 train: -0.6298280569640073 val: -0.617947906255722
epoch: 15 train: -0.6288465342738412 val: -0.6172269620001316
epoch: 16 train: -0.6280093285170468 val: -0.616608764976263
epoch: 17 train

In [122]:
epochs = 100
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    mid_seperate_encoder.train()
    for x, y in train_mid_dateloader:
        Z_x, Z_y = mid_seperate_encoder(x, y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        mid_seperate_optimizer.zero_grad()
        loss.backward()
        mid_seperate_optimizer.step()
        with torch.no_grad():
            current_weights = mid_seperate_encoder.current.weight
            future_weights = mid_seperate_encoder.future.weight
            
            current_weights.div_(current_weights.norm(p=2)) 
            future_weights.div_(future_weights.norm(p=2))
             
    val_loss = 0
    mid_seperate_encoder.eval()
    for x, y in val_mid_dateloader:
        with torch.no_grad():
            Z_x, Z_y = mid_seperate_encoder(x, y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_mid_dateloader)} val: {val_loss / len(val_mid_dateloader)}")   
            
    avg_val_loss = val_loss / len(val_mid_dateloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        best_state = copy.deepcopy(mid_seperate_encoder.state_dict())
    
if best_state is not None:
    mid_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.14858347895470533 val: -0.20272965356707573
epoch: 2 train: -0.46299759948795494 val: -0.5983281135559082
epoch: 3 train: -0.7255121924660423 val: -0.7484617494046688
epoch: 4 train: -0.8029832937500694 val: -0.7935477085411549
epoch: 5 train: -0.8255396268584512 val: -0.8085761591792107
epoch: 6 train: -0.8323849103667519 val: -0.8134831823408604
epoch: 7 train: -0.8339550766077909 val: -0.8145454525947571
epoch: 8 train: -0.8336110841144215 val: -0.8140841946005821
epoch: 9 train: -0.8325930898839777 val: -0.8130499869585037
epoch: 10 train: -0.8313944751566107 val: -0.8118547052145004
epoch: 11 train: -0.8302155624736439 val: -0.8106769919395447
epoch: 12 train: -0.8291336027058688 val: -0.809591393917799
epoch: 13 train: -0.8281720399856567 val: -0.8086223304271698
epoch: 14 train: -0.8273308244618502 val: -0.8077719360589981
epoch: 15 train: -0.8266002817587419 val: -0.8070319704711437
epoch: 16 train: -0.8259674679149281 val: -0.8063902817666531
epoch: 17 train

In [123]:
with torch.no_grad():
    Z_x = mid_shared_encoder(test_current_tensor)
    Z_y = mid_shared_encoder(test_future_mid_tensor)
    mid_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {mid_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = mid_seperate_encoder(test_current_tensor, test_future_mid_tensor)
    mid_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {mid_sep_loss}")

shared loss: -0.6555103659629822
seperate loss: -0.8236406445503235


In [124]:
mid_test_result = mid_shared_loss / mid_sep_loss
print(f"Mid test result: {mid_test_result}, Mid shareability result: {mid_shareability}")

Mid test result: 0.7958693504333496, Mid shareability result: 0.7611133262980686


## High Rank One Case

In [125]:
high_shared_encoder = SharedEncoder(vector_size=13)
high_shared_optimizer = torch.optim.Adam(high_shared_encoder.parameters(), lr=1e-2)

high_seperate_encoder = SeperateEncoder(vector_size=13)
high_seperate_optimizer = torch.optim.Adam(high_seperate_encoder.parameters(), lr=1e-2)

In [126]:
epochs = 100
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    high_shared_encoder.train()
    for x, y in train_high_dateloader:
        Z_x = high_shared_encoder(x)
        Z_y = high_shared_encoder(y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        high_shared_optimizer.zero_grad()
        loss.backward()
        high_shared_optimizer.step()
        with torch.no_grad():
            weights = high_shared_encoder.shared.weight
            weights.div_(weights.norm(p=2))
             
    val_loss = 0
    high_shared_encoder.eval()
    for x, y in val_high_dateloader:
        with torch.no_grad():
            Z_x = high_shared_encoder(x)
            Z_y = high_shared_encoder(y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_high_dateloader)} val: {val_loss / len(val_high_dateloader)}")   
            
    avg_val_loss = val_loss / len(val_high_dateloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        best_state = copy.deepcopy(high_shared_encoder.state_dict())

        
    
if best_state is not None:
    high_shared_encoder.load_state_dict(best_state)

epoch: 1 train: -0.23581304096362807 val: -0.4038166329264641
epoch: 2 train: -0.5400112515146082 val: -0.6330984346568584
epoch: 3 train: -0.7104314283891158 val: -0.7122495919466019
epoch: 4 train: -0.7568222739479759 val: -0.7299043871462345
epoch: 5 train: -0.7617612892931158 val: -0.7268005311489105
epoch: 6 train: -0.7537687670100819 val: -0.7168186753988266
epoch: 7 train: -0.7421550371430137 val: -0.7050882205367088
epoch: 8 train: -0.730224871635437 val: -0.6936193853616714
epoch: 9 train: -0.7191917007619685 val: -0.6831999868154526
epoch: 10 train: -0.7094732658429579 val: -0.6741053089499474
epoch: 11 train: -0.7011646742170508 val: -0.6663839891552925
epoch: 12 train: -0.6942237285050479 val: -0.6599801331758499
epoch: 13 train: -0.6885490601713007 val: -0.654790285974741
epoch: 14 train: -0.6840143176642332 val: -0.6506884433329105
epoch: 15 train: -0.6804849299517545 val: -0.6475430876016617
epoch: 16 train: -0.6778270212086764 val: -0.6452224999666214
epoch: 17 train: -

In [127]:
epochs = 100
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    high_seperate_encoder.train()
    for x, y in train_high_dateloader:
        Z_x, Z_y = high_seperate_encoder(x, y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        high_seperate_optimizer.zero_grad()
        loss.backward()
        high_seperate_optimizer.step()
        with torch.no_grad():
            current_weights = high_seperate_encoder.current.weight
            future_weights = high_seperate_encoder.future.weight
            
            current_weights.div_(current_weights.norm(p=2)) 
            future_weights.div_(future_weights.norm(p=2))
             
    val_loss = 0
    high_seperate_encoder.eval()
    for x, y in val_high_dateloader:
        with torch.no_grad():
            Z_x, Z_y = high_seperate_encoder(x, y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_high_dateloader)} val: {val_loss / len(val_high_dateloader)}")   
            
    avg_val_loss = val_loss / len(val_high_dateloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        best_state = copy.deepcopy(high_seperate_encoder.state_dict())

    
if best_state is not None:
    high_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.1028919130563736 val: -0.06906462530605495
epoch: 2 train: -0.13011222774670883 val: -0.06086635496467352
epoch: 3 train: -0.1450454377653924 val: -0.06482202652841806
epoch: 4 train: -0.15289567245001143 val: -0.06792866066098213
epoch: 5 train: -0.1581332968039946 val: -0.07076621148735285
epoch: 6 train: -0.16408029445870356 val: -0.06976651959121227
epoch: 7 train: -0.16888601786711 val: -0.07261357922106981
epoch: 8 train: -0.17134688858958808 val: -0.07347804598975927
epoch: 9 train: -0.17223103642463683 val: -0.07354943361133337
epoch: 10 train: -0.17219341502270916 val: -0.0693304289598018
epoch: 11 train: -0.17395566414025695 val: -0.06763867067638785
epoch: 12 train: -0.17617586332965982 val: -0.08392075938172638
epoch: 13 train: -0.18206021751869808 val: -0.09938407223671675
epoch: 14 train: -0.19341860472817313 val: -0.12264181091450155
epoch: 15 train: -0.33991619511084126 val: -0.5906848609447479
epoch: 16 train: -0.7686867545951497 val: -0.824374623596

In [128]:
with torch.no_grad():
    Z_x = high_shared_encoder(test_current_tensor)
    Z_y = high_shared_encoder(test_future_high_tensor)
    high_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {high_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = high_seperate_encoder(test_current_tensor, test_future_high_tensor)
    high_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {high_sep_loss}")

shared loss: -0.7141880393028259
seperate loss: -0.8635461926460266


In [129]:
high_test_result = high_shared_loss / high_sep_loss
print(f"High test result: {high_test_result}, High shareability result: {high_shareabilitty}")

High test result: 0.8270409107208252, High shareability result: 0.9992489480932957
